# **2. Deep Q-Learning para _cliff walking_**

En este cuaderno, vamos a resolver de nuevo el problema de [Cliff Walking](https://gymnasium.farama.org/environments/toy_text/cliff_walking/), pero sustituyendo la tabla $Q$ de [01_cliffwalking-qlearning.ipynb](01_cliffwalking-qlearning.ipynb) por una red neuronal que aproxima $Q(s, a)$: **Deep Q-Learning** (DQN).

_Considere el mundo de cuadrícula ("gridworld") mostrado abajo. Es una tarea episódica estándar, con un estado inicial (casilla S) y meta (casilla G). Las acciones posibles para moverse son arriba, abajo, derecha e izquierda. La recompensa es -1 en todas las transiciones, salvo en las que llevan a la región marcada como «El Acantilado» ("The Cliff"). Pisar esta región supone una recompensa de -100 y devuelve al agente instantáneamente al estado inicial._

<img src="../../img/cliffwalking.png" width=500/>

Basado en:
> R.S. Sutton, A.G. Barto (2018) Reinforcement Learning. Capítulos 6.5: Q-learning: control TD _off-policy_, 6.6: _Expected Sarsa_.

> Udacity, Deep Reinforcement Learning Course. Disponible en [GitHub](https://github.com/udacity/deep-reinforcement-learning/tree/master/dqn).

## **2.0. Preparación**

Comenzamos importando las bibliotecas necesarias, así como otras utilidades que emplearemos más adelante:

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import TransformObservation
import numpy as np
import torch
from collections import deque

import io
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, Image as IPImage
from PIL import Image

def show_state(env):
    """
    Muestra el estado actual del entorno.
    """
    img = env.render()
    clear_output(wait=True)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

def show_episode(env, act):
    """
    Ejecuta un episodio completo y lo muestra como una animación.

    Parámetros
    ==========
        env: entorno creado con render_mode="rgb_array"
        act (callable): función que devuelve la acción a realizar en un estado
    """
    state, info = env.reset()
    frames = [Image.fromarray(env.render())]
    score = 0
    terminated = truncated = False

    # Bucle de interacción agente-entorno (1 episodio)
    while not (terminated or truncated):
        action = act(state)
        state, reward, terminated, truncated, info = env.step(action)
        frames.append(Image.fromarray(env.render()))
        score += reward

    # Guardar las imágenes como un GIF animado y mostrarlo
    gif = io.BytesIO()
    frames[0].save(gif, format="gif", save_all=True, append_images=frames[1:],
                   duration=1000 // env.metadata["render_fps"], loop=0)
    display(IPImage(data=gif.getvalue(), format="gif"))
    print("Puntuación: ", score)

def plot_scores(scores, title, solved_score=None, window=100):
    """
    Representa la puntuación de cada episodio de entrenamiento y la media móvil de los últimos `window`.

    Parámetros
    ==========
        scores (list): puntuación de cada episodio de entrenamiento
        title (str): título de la gráfica
        solved_score (float): puntuación media para considerar resuelto el problema (opcional)
        window (int): número de episodios de la media móvil
    """
    scores = np.asarray(scores)
    avg = [np.mean(scores[max(0, i-window+1):i+1]) for i in range(len(scores))]
    plt.plot(np.arange(len(scores)), scores, alpha=0.4, label="Puntuación")
    plt.plot(np.arange(len(scores)), avg, label=f"Puntuación media (últimos {window})")
    if solved_score is not None:
        plt.axhline(solved_score, color="gray", linestyle="--", label="Resuelto")
    plt.ylabel("Puntuación")
    plt.xlabel("Episodio nº")
    plt.title(title)
    plt.legend()
    plt.show()

def print_policy(actions):
    """
    Muestra la política como una cuadrícula de flechas (C: acantilado, T: meta).

    Parámetros
    ==========
        actions (array_like): acción de cada uno de los 48 estados (ARRIBA = 0, DERECHA = 1, ABAJO = 2, IZQUIERDA = 3)
    """
    grid = np.array(["↑", "→", "↓", "←"])[np.asarray(actions)].reshape(4, 12)
    grid[3, 1:11] = "C"   # Celdas del acantilado
    grid[3, 11] = "T"     # Meta
    print("\n".join(" ".join(row) for row in grid))

def plot_values(V):
    """
    Representa la función de valor de estado sobre la cuadrícula de 4 x 12 celdas.

    Parámetros
    ==========
        V (array_like): valor de cada uno de los 48 estados
    """
    V = np.reshape(V, (4, 12))
    fig = plt.figure(figsize=(15, 5))
    ax = fig.add_subplot(111)
    ax.imshow(V, cmap="cool")
    for (j, i), label in np.ndenumerate(V):
        ax.text(i, j, f"{label:.1f}", ha="center", va="center", fontsize=14)
    plt.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
    plt.title("Función de valor de estado")
    plt.show()

## **2.1. Entorno**

Queremos reutilizar, sin ningún cambio, el mismo agente DQN que usaremos para [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/) en [03_cartpole-dqn.ipynb](03_cartpole-dqn.ipynb). Esto requiere dos adaptaciones del entorno:

*   El estado de CliffWalking es el índice de la celda en la que se encuentra el agente (un entero de $0$ a $47$), pero la red Q espera un vector de números. Transformamos cada estado en un vector _one-hot_ de tamaño $48$ (todo ceros, salvo un $1$ en la posición de la celda) con el _wrapper_ [`TransformObservation`](https://gymnasium.farama.org/api/wrappers/observation_wrappers/#gymnasium.wrappers.TransformObservation).
*   Caer por el acantilado no termina el episodio, así que una mala política puede caminar indefinidamente. Limitamos la longitud de los episodios a $100$ pasos con el parámetro `max_episode_steps`: al alcanzar ese límite, el episodio se trunca (`truncated = True`).

In [ ]:
def one_hot(state):
    """Codifica el índice de una celda (0-47) como un vector one-hot de tamaño 48."""
    return np.eye(48, dtype=np.float32)[state]

env = gym.make("CliffWalking-v1", render_mode="rgb_array", max_episode_steps=100)
env = TransformObservation(env, one_hot, gym.spaces.Box(0, 1, (48,), np.float32))

print(env.observation_space)
print(env.action_space)

Una vez creado el entorno, podemos visualizar el **estado inicial** de la simulación:

In [ ]:
# Inicializar el entorno
env.reset()

# Mostrar estado inicial del entorno
show_state(env)

Podemos interactuar con el entorno con un agente sin entrenar usando la clase `Agent` de [dqn_agent.py](dqn_agent.py), una utilidad adicional que hemos implementado. La única diferencia con CartPole es el tamaño del estado ($48$) y el número de acciones ($4$). La función `show_episode()` muestra el episodio como una animación, junto con su **puntuación**, es decir, la suma de las recompensas obtenidas a lo largo del episodio:

In [ ]:
from dqn_agent import Agent

agent_untrained = Agent(state_size=48, action_size=4, seed=0)

show_episode(env, agent_untrained.act)

## **2.2. Algoritmo DQN**

A continuación, usamos la misma implementación de Deep Q-Learning (DQN) que en [03_cartpole-dqn.ipynb](03_cartpole-dqn.ipynb). Nos apoyamos en ficheros externos:

*   [dqn_agent.py](dqn_agent.py): Implementación de las capacidades del agente (`act`, `step`, etc.)
*   [model.py](model.py): Implementación de la red neuronal del agente (la red Q)

La política óptima (caminar por el borde del acantilado) obtiene una puntuación de $-13$. Sin embargo, durante el entrenamiento el agente sigue tomando algunas acciones aleatorias ($\epsilon$-_greedy_), que ocasionalmente le hacen caer por el acantilado. Por ello, consideramos el problema resuelto cuando el agente obtiene una puntuación media de al menos $-20$ puntos en 100 episodios consecutivos.

In [ ]:
SOLVED_SCORE = -20.0   # Puntuación media (últimos 100 episodios) para considerar resuelto CliffWalking-v1

def dqn(agent, n_episodes=2000, eps_start=1.0, eps_end=0.01, eps_decay=0.995, print_every=100):
    """Deep Q-Learning.

    Parámetros
    ==========
        agent (Agent): agente a entrenar
        n_episodes (int): número máximo de episodios
        eps_start (float): valor inicial de epsilon
        eps_end (float): valor final de epsilon
        eps_decay (float): factor de multiplicación de epsilon (por episodio)
        print_every (int): mostrar la puntuación cada print_every episodios
    """
    scores = []                        # Puntuaciones de todos los episodios
    scores_window = deque(maxlen=100)  # Puntuaciones de los últimos 100 episodios
    eps = eps_start                    # Inicializar epsilon

    for i_episode in range(1, n_episodes+1):
        state, info = env.reset()
        score = 0
        terminated = truncated = False

        # Bucle de interacción agente-entorno (1 episodio)
        while not (terminated or truncated):
            # Elegir la acción At con la política ε-greedy
            action = agent.act(state, eps)

            # Aplicar At y obtener Rt+1, St+1
            next_state, reward, terminated, truncated, info = env.step(action)

            # Almacenar <St, At, Rt+1, St+1>, entrenar y actualizar
            # (solo 'terminated' marca St+1 como terminal; tras un truncamiento hay que seguir haciendo bootstrapping)
            agent.step(state, action, reward, next_state, terminated)

            # Avanzar al siguiente estado
            state = next_state
            score += reward

        scores_window.append(score)       # Guardar la última puntuación (ventana de 100)
        scores.append(score)              # Guardar la última puntuación (todos los episodios)
        eps = max(eps_end, eps_decay*eps) # Reducir epsilon

        # Mostrar información del entrenamiento
        if i_episode % print_every == 0:
            print(f"Episodio {i_episode}\tPuntuación media (últimos 100): {np.mean(scores_window):.2f}")
        if len(scores_window) == 100 and np.mean(scores_window) >= SOLVED_SCORE:
            print(f"\n¡Problema resuelto en {i_episode-100} episodios!\tPuntuación media (últimos 100): {np.mean(scores_window):.2f}")
            break
    else:
        print(f"\nProblema no resuelto en {n_episodes} episodios.\tPuntuación media (últimos 100): {np.mean(scores_window):.2f}")

    torch.save(agent.qnetwork_local.state_dict(), "checkpoint-cliffwalking-dqn.pth") # Guardar los pesos del agente entrenado

    return scores

agent = Agent(state_size=48, action_size=4, seed=0)
scores = dqn(agent)

plot_scores(scores, "DQN", SOLVED_SCORE)

## **2.3. Política aprendida y función de valor de estado**

Como en [01_cliffwalking-qlearning.ipynb](01_cliffwalking-qlearning.ipynb), podemos mostrar la política aprendida (_greedy_). Aquí los valores $Q(s, a)$ no se leen de una tabla, sino que los calcula la red Q (`agent.qnetwork_local`) para los vectores _one-hot_ de los 48 estados.

In [ ]:
states = torch.eye(48)   # Vectores one-hot de los 48 estados
with torch.no_grad():
    Q = agent.qnetwork_local(states).numpy()   # Valores Q de los 48 estados, forma (48, 4)

print_policy(Q.argmax(axis=1))

Podemos visualizar los valores de la función de valor de estado $v_\pi$ para la política estimada $\pi$ y compararlos con los valores de la política óptima $v_{\pi_*}$. Nótese que [dqn_agent.py](dqn_agent.py) usa $\gamma = 0.99$ en lugar de $\gamma = 1$, por lo que los valores de los estados lejanos a la meta son ligeramente mayores (menos negativos) que los óptimos.

__Función de valor de estado de la política aprendida por el agente $v_\pi$__

In [ ]:
V = Q.max(axis=1)
V[37:] = 0   # Las celdas del acantilado y de la meta nunca se visitan
plot_values(V)

__Función de valor de estado de la política óptima $v_{\pi_*}$__

In [ ]:
# Los valores de estado de la política óptima pueden calcularse manualmente
V_opt = np.zeros((4, 12))
V_opt[0][:] = -np.arange(3, 15)[::-1]
V_opt[1][:] = -np.arange(3, 15)[::-1] + 1
V_opt[2][:] = -np.arange(3, 15)[::-1] + 2
V_opt[3][0] = -13
plot_values(V_opt)

## **2.4. Visualizar el agente entrenado**

Cargamos los pesos en el agente y lo ejecutamos durante un episodio con la política _greedy_:

In [ ]:
# Cargar los pesos desde `checkpoint-cliffwalking-dqn.pth`
agent_load = Agent(state_size=48, action_size=4, seed=0)

agent_load.qnetwork_local.load_state_dict(torch.load("checkpoint-cliffwalking-dqn.pth", weights_only=True))

show_episode(env, agent_load.act)

¡Nuestro agente ha aprendido de nuevo a cruzar el acantilado, esta vez con una red neuronal!

En este cuaderno, hemos sustituido la tabla $Q$ por una red neuronal y hemos visto cómo adaptar el entorno (codificación _one-hot_ y límite de pasos) para reutilizar un agente DQN genérico.

En el próximo cuaderno, aplicaremos el mismo agente a un problema con estados continuos, donde una tabla ya no es viable.